<a href="https://colab.research.google.com/github/DanDC25/Statistical-Learning-with-Python/blob/main/%5BPractice_LEAKAGE_Case_Study%5D_DSA211_Simulated_Q4_Case_Study.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## How to use this question

This case is built from **Lab 7, Part 5 (Data Leakage: Seeing It, Then Fixing It)**, with points from **Lab 8** and **Lab 9**. The analyst's code is the lab's "wrong way" demo with a few more faults added.

- Work like the exam: about **18 minutes**, open book, no AI. **No code is needed.**
- Answer each part in the markdown cell under its label. Cite line numbers.
- Keep to the sentence limits. Answers are marked for precision.
- The **📍 Where to look** note under each part says which lab or lecture section covers it. Try first, then look.

***

### Background

A marketing team ran an email campaign to **50 customers**. About half responded. For each customer the team has **1,000 web-behaviour features** (page views, clicks, time on site, and so on). An analyst was asked whether these features can predict who responds.

| Column | Description |
|:---|:---|
| `responded` | 1 if the customer responded to the campaign. **The target.** |
| 1,000 other columns | Web-behaviour features, all numeric |

: Data dictionary

No data are provided. All outputs shown are those reported by the analyst.

***

### The analyst's code

```text
 1  import numpy as np
 2  import pandas as pd
 3  from sklearn.preprocessing import StandardScaler
 4  from sklearn.feature_selection import SelectKBest, f_classif
 5  from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
 6  from sklearn.linear_model import LogisticRegression
 7  from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
 8  from sklearn.naive_bayes import GaussianNB
 9  from sklearn.metrics import accuracy_score
10  
11  df = pd.read_csv("campaign.csv")              # 50 customers, 1,000 features
12  X = df.drop(columns="responded").values
13  y = df["responded"].values
14  
15  # Standardise all predictors
16  X = StandardScaler().fit_transform(X)
17  
18  # Keep the 20 predictors most associated with the response
19  selector = SelectKBest(f_classif, k=20)
20  X_sel = selector.fit_transform(X, y)
21  
22  X_tr, X_te, y_tr, y_te = train_test_split(X_sel, y, test_size=0.3, stratify=y,
23                                            random_state=2026)
24  
25  skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=2026)
26  scores = cross_val_score(LogisticRegression(), X_tr, y_tr, cv=skf)
27  se = scores.std() / np.sqrt(5)
28  print(f"Logistic CV accuracy: {scores.mean():.2f} (SE {se:.2f})")
29  
30  models = {'Logistic': LogisticRegression(),
31            'LDA': LinearDiscriminantAnalysis(),
32            'Naive Bayes': GaussianNB()}
33  for name, model in models.items():
34      model.fit(X_tr, y_tr)
35      print(f"{name:12s} test accuracy: {accuracy_score(y_te, model.predict(X_te)):.2f}")
```

| Output |
|:---|
| `Logistic CV accuracy: 0.94 (SE 0.03)` |
| `Logistic     test accuracy: 0.93` |
| `LDA          test accuracy: 0.93` |
| `Naive Bayes  test accuracy: 0.93` |

: Output of the code

The analyst's report concludes: *"Logistic regression predicts campaign response with 93% accuracy on customers it never saw. LDA and Naive Bayes do equally well, which confirms that the 20 selected features are robust."*

***

**4(a)** Look only at the background and the output. Give two reasons why an experienced analyst would doubt these results before reading a line of the code. **[2]** *(at most 3 sentences)*

**4(a)**

*Your answer here:*

It appears that the dataset proportion of number of observations and numbers of predictor are extremely imbalanced, specifically, the set-up was 50 customers and 1000 predictors. Multiple-testing problem states that the more predictors you have, the more likely to have 1 or more false rejection simply by chance alone. In this context, by running 1000 F-tests on few observations (50), it is likely that some of these predictors, by chance alone, appear as if they relate to the labels. Moreover, it seems suspicious that all output numbers are similar to each other, if not even identical; this raises a flag that the code may have mistakenly only printed 1 model's output rather than all models' to compare. (Check 4d line 26)

> 📍 **Where to look**
>
> - Lab 7, **Part 5** introduction: the set-up with $n = 50$ and $p = 1000$, and what accuracy a method can honestly reach when half the customers respond.
> - Lab 7, **Question 5.1a**: what 1,000 tests on 50 observations guarantee. This is the Week 3 multiple-testing problem.

***

**4(b)** One line of the code explains the output that 4(a) doubts. Give its line number and explain how it produces that output. **[2]** *(at most 3 sentences)*



**4(b)**

*Your answer here:*

I believe that it is line 20, an attempt to select all 20 predictors out of 1000 predictors on the entire dataset to "fix" the problem stated in question 4(a). However, this leads to leakages as leakage (from selecting before cross-validation without using Pipeline) had entered before CV-split in preprocessing. Selecting the "winners" predictor, the chosen predictor partially memorizes the outcome including the outcome of every future held-out fold. By the time CV is ran, the held-out customers were no longer hidden, so, the CV accuracy is inflated.



> 📍 **Where to look**
>
> - Lab 7, **Part 5**, the first demo cell (the comment `THE WRONG WAY`). Find the same step in the analyst's code.
> - Lab 7, **Question 5.1a**: why selecting the "winners" contaminates every held-out fold.

***

**4(c)** The analyst defends the result: "The 15 test customers were never used to fit any model, so 0.93 is an honest estimate." Respond. **[2]** *(at most 3 sentences)*

**4(c)**

*Your answer here:*

No, even when the 15 test customers were never used to fit any model, the 0.93 accuracy is still inflated. This is because selecting the winners before running CV creates data-leakages where the "winner" predictor already partially remembers the outcome of every future held-out fold, even the 15 test customer. Hence, when CV is ran, even when the 15 test customer is not touched, the held-out customers are no longer hidden as the predictors "remember" them.

> 📍 **Where to look**
>
> - Compare the line numbers of the selection step and the split. Which comes first?
> - Lecture 7, **Section 8: Data Leakage: Preprocessing Belongs Inside the Loop**, and **8.1 Demo: Leakage in Action**.

***

**4(d)** Apart from the fault in 4(b), the code contains further faults. Identify **two** of them by line number, say why each is wrong, and say what the code should do instead. **[2]** *(at most 3 sentences)*

**4(d)**

*Your answer here:*

The two further faults are in line 16 and line 26.

Line 16: fitting StandardScaler on **all** of X computes mean and standard deviation of every fold, including the non-touched test fold that leads to data leakages and an overly optimistic CV score; what the analyst should've done is fit the scaler inside a Pipeline then Cross-Validate, now, the scaler happens inside each training fold. Hence, the code should return a cross-validation score that prevents data leakage.

Line 26: It's apparent that all models (Logistic Regression, LDA, Naive Bayes) all have the same CV accuracy; this is because the variable scores wasn't made inside of the loop and appended, it only returns the CV accuracy of the Logistic Regression that it was made for. The analyst should create a list that stores scores when it is appended in the loop comparing the 3 models. Hence, in doing so, the code should return the CV accuracy of all models, not just of 1.

> 📍 **Where to look**
>
> - **Preprocessing:** Lab 7, **Question 5.1b**, item (b) (standardising with the sample mean and SD: inside the fold or outside?).
> - **Standard error:** Lab 8, **Part 4**, the `cv_se` line (`ddof=1`), and Lab 9, **3.1 The CV Error Curve** (why `sklearn` gives `ddof=0`).
> - **The test set:** Lab 9 introduction (touch the test set once) and **Exercise 4.2**. How many models does the analyst score on it, and what does the report then conclude?

***

**4(e)** A colleague puts the scaling, the selection and the logistic regression into one `Pipeline`, splits the raw data first, and reruns. Her output: CV accuracy **0.46** (SE 0.07), test accuracy **0.47**. Which result do you believe, what would you report to the team, and what is the general rule? **[2]** *(at most 3 sentences)*

**4(e)**

*Your answer here:*

Now that scaling, selection, and the logistic regression is put into one Pipeline, I believe the CV accuracy and test accuracy because pre-processing has become a part of the model. However, I would report the test accuracy as it was tested on a new, unseen data that provides an unbiased estimate of how the model generalizes on unseen data in reality.

The general rule is that steps that allow the model to learn the slightest, is disallowed to avoid leakage; as pre-processing is a common cause towards data-leakage, pre-processing should be made part of the model by utilizing sklearn's Pipeline to pre-process data inside each training fold.


> 📍 **Where to look**
>
> - Lab 7, **Exercise 5.1: Move the selection inside the pipeline**, and the result it prints.
> - Lab 7, **Question 5.1b** (state the general rule), and Lecture 7, **8.2 The Fix: Make Preprocessing Part of the Model**.

***

## After you finish

1. Did every answer cite a line number or a figure from the output?
2. Could you have found the 4(b) fault without the pointer? If not, rerun Lab 7, Part 5 until you can explain it in two sentences.